# Part 1 — Data Exploration(EDA)

**Dataset:** US Flight Delays 2015 (3 files in `data/`):
- `flights.csv` — main table (~5.8M flights)
- `airlines.csv` — airline code → name
- `airports.csv` — airport code → city, state, coordinates

**Goal:** understand the data and decide:
- how to define "delayed" (target);
- which columns can be used as features;
- which columns must not be used (data leakage);
- identify data attributes for further BI analytics.

##Dataset issues:

The original dataset contains approximately 5.8 million flight records for 2015. Due to Databricks upload limitations, this project uses a reduced dataset of 200,000 records, covering January 1–13, 2015.

In another words, findings from this dataset should not be generalized to all US flights in 2015 without further validation.

In [0]:
import pandas as pd
from pathlib import Path

#main path
DATA_DIR = Path("../data")

In [0]:
# flights.csv is very large (~500 MB), so a lower sized sample created.
SAMPLE_ROWS = 200_000
#===========================================================
flights = pd.read_csv(DATA_DIR / "flights_sample.csv", nrows=SAMPLE_ROWS)
airlines = pd.read_csv(DATA_DIR / "airlines.csv")
airports = pd.read_csv(DATA_DIR / "airports.csv")

print(f"flights:  {flights.shape[0]:,} rows, {flights.shape[1]} columns (sample)")
print(f"airlines: {airlines.shape[0]} airlines")
print(f"airports: {airports.shape[0]} airports")

## 1. First look

In [0]:
flights.head()

In [0]:
print("Columns:")
print(flights.columns.tolist())

In [0]:
flight_dates = pd.to_datetime(
    flights[["YEAR", "MONTH", "DAY"]].rename(
        columns={"YEAR": "year", "MONTH": "month", "DAY": "day"}
    )
)

print("Earliest flight:", flight_dates.min())
print("Latest flight:", flight_dates.max())

##Attention:

Date limitations should be considered when interpreting both ML model performance and future BI dashboards, since dataset doesn't capture flight patterns from other months or seasons.

## 2. Missing values and types

In [0]:
missing = flights.isnull().sum().sort_values(ascending=False)
missing[missing > 0]

In [0]:
missing_percentage = (
    flights.isnull().mean() * 100
).sort_values(ascending=False)

missing_percentage[missing_percentage > 0].round(2)

In [0]:
flights.dtypes

In [0]:
flights.info()

## 3. Delay-related columns

In [0]:
delay_cols = ["DEPARTURE_DELAY",
    "ARRIVAL_DELAY", "TAXI_OUT",
    "TAXI_IN", "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY", "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY",]
#needed for data analysis
flights[delay_cols].describe()

**Observation:** The average departure delay is ~16.95 minutes, median is only 1 minute. This means some very long delays increase the average val.

Negative delays mean a flight departed early(ig so).

**TAXI_OUT** and **TAXI_IN** show time spent moving on the ground, not any type of delays.

## 4. Target — delayed or not

**Target:** We want to predict whether a flight will depart at least 15 minutes late.

- **1** — delay is 15 minutes or more.
- **0** — delay is less than 15 minutes.
- **NA** — departure delay is unknown.

We use **DEPARTURE_DELAY** to create the target.

In [0]:
flights["is_delayed"] = (
    flights["DEPARTURE_DELAY"] >= 15).where(
        flights["DEPARTURE_DELAY"].notna()
    ).astype("Int64")
#at least 15 mins implies delayed
print(flights["is_delayed"].value_counts(dropna=False))
print()
print("Share of delayed flights:")
print("\tAmong known:")
print(flights["is_delayed"].value_counts(normalize=True).round(3))
print()
print("\tAmong all:")
print(flights["is_delayed"].value_counts(normalize=True, dropna=False).round(3))
print()
print("Unknown target:", flights["is_delayed"].isna().sum())

Among flights with known departure delays, 70.8% weren't delayed and 29.2% were delayed.
Another 4,868 flights have unknown delays. non-delayed flights > delayed flights.

## 5. Cancelled and diverted flights

In [0]:
print("Cancelled (CANCELLED):", flights["CANCELLED"].value_counts().to_dict())
print("Diverted (DIVERTED):", flights["DIVERTED"].value_counts().to_dict())

5,077 cancelled flights and 522 diverted flights.

We may remove these flights when preparing data for ML, but we should keep them in Silver, because they are useful for BI dashboards.

## 6. Lookup tables: airlines and airports

In [0]:
airlines.head()

In [0]:
airports.head()

In [0]:
print("Duplicated airline codes")
print(airlines["IATA_CODE"].duplicated().sum())
print()
print("Duplicated airport codes")
print(airports["IATA_CODE"].duplicated().sum())

**Validated:** Airline and airport codes have no duplicates or missing values.

We still need to check if all codes in flights table exist in these lookup tables.

In [0]:
print("Missing airline codes:", airlines["IATA_CODE"].isna().sum())
print("Missing airport codes:", airports["IATA_CODE"].isna().sum())

This means, key values are unique and complete

## 7. Which columns can / cannot be used

We should only use information, that is available before the flight departs. Using information about what actually happened during or after the flight can cause data leakage. These rules apply to ML predictions, not to BI dashboards.

| Column | OK? | Why |
|--------|-----|-----|
| YEAR, MONTH, DAY, DAY_OF_WEEK | yes | known in advance |
| AIRLINE | yes | known before departure |
| FLIGHT_NUMBER | yes | known before departure |
| ORIGIN_AIRPORT, DESTINATION_AIRPORT | yes | route known in advance |
| SCHEDULED_DEPARTURE, SCHEDULED_ARRIVAL | yes | schedule known in advance |
| DISTANCE | yes | route distance known in advance |
| DEPARTURE_DELAY, ARRIVAL_DELAY | **no** | this is the delay (target or leakage) |
| DEPARTURE_TIME, ARRIVAL_TIME | **no** | actual times — only known after |
| TAXI_OUT, WHEELS_OFF, WHEELS_ON, TAXI_IN | **no** | happens during/after the flight |
| ELAPSED_TIME, AIR_TIME | **no** | only known after the flight |
| AIR_SYSTEM_DELAY, WEATHER_DELAY, ... | **no** | delay reasons — after the delay happened |
| CANCELLED, DIVERTED | **no** | flight outcome, not a pre-departure feature |
| TAIL_NUMBER | mb, not sure | usually skipped at the start |

## 8. Summary (Part 1)

1. **Data:** 200,000 flights from January 1–13, 2015.
2. **Target:** A flight is delayed if **DEPARTURE_DELAY >= 15**. Missing delays remain unknown.
3. **Classes:** 70.8% Not Delayed and 29.2% Delayed among flights with known delays.
4. **Missing Values:** Some missing values are normal and shouldn't always be removed.
5. **ML Features:** Use only information available before departure.
6. **Lookup Tables:** No duplicate or missing airline and airport codes.
7. **Next Steps:** Build Bronze and Silver, then prepare data for ML and BI.